In [1]:

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from scipy import stats

from google.colab import files
uploaded = files.upload()




Saving TeachingRatings.csv to TeachingRatings.csv


In [ ]:
df = pd.read_csv("TeachingRatings.csv")
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

Dataset shape: (463, 13)

Column names:
['rownames', 'minority', 'age', 'gender', 'credits', 'beauty', 'eval', 'division', 'native', 'tenure', 'students', 'allstudents', 'prof']

First five rows:


,rownames,minority,age,gender,credits,beauty,eval,division,native,tenure,students,allstudents,prof
0,1,yes,36,female,more,0.289916,4.3,upper,yes,yes,24,43,1
1,2,no,59,male,more,-0.737732,4.5,upper,yes,yes,17,20,2
2,3,no,51,male,more,-0.571984,3.7,upper,yes,yes,55,55,3
3,4,no,40,female,more,-0.677963,4.3,upper,yes,yes,40,46,4
4,5,no,31,female,more,1.509794,4.4,upper,yes,yes,42,48,5


In [ ]:
df['female'] = (
    df['gender'].astype(str).str.strip().str.lower() == 'female'
).astype(int)
model_q1 = smf.ols('eval ~ female', data=df).fit()
print(model_q1.summary())

print("\nGender Coefficient: ", model_q1.params['female'])
print("t-statistic: ", model_q1.tvalues['female'])
print("p-value: ", model_q1.pvalues['female'])





                            OLS Regression Results                            
Dep. Variable:                   eval   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     10.56
Date:                Wed, 14 Oct 2026   Prob (F-statistic):            0.00124
Time:                        04:12:54   Log-Likelihood:                -378.50
No. Observations:                 463   AIC:                             761.0
Df Residuals:                     461   BIC:                             769.3
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      4.0690      0.034    121.288      0.0

The regression coefficient for female instructors is approximately −0.1680, indicating that female instructors have an average teaching evaluation score about 0.168 points lower than male instructors in this model. The p-value is approximately 0.001, which is less than 0.05. Therefore, we reject the null hypothesis and conclude that gender is statistically significantly associated with teaching evaluation scores in this dataset.

Gender has a statistically significant association with teaching evaluation scores.

In [ ]:

print("Age summary:")
print(df['age'].describe())

print("\nUnique ages:")
print(sorted(df['age'].dropna().unique()))

print("\nAge quartiles:")
print(df['age'].quantile([0.25, 0.50, 0.75]))


Age summary:
count    463.000000
mean      48.365011
std        9.802742
min       29.000000
25%       42.000000
50%       48.000000
75%       57.000000
max       73.000000
Name: age, dtype: float64

Unique ages:
[np.int64(29), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(54), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(70), np.int64(73)]

Age quartiles:
0.25    42.0
0.50    48.0
0.75    57.0
Name: age, dtype: float64


In [ ]:

df['age_group'] = pd.cut(
    df['age'],
    bins=[-float('inf'), 41, 57, float('inf')],
    labels=['Young', 'Middle-aged', 'Older'],
    right=False
)

model_q2 = smf.ols(
    'beauty ~ C(age_group)', data=df
).fit()

anova_results = anova_lm(model_q2, typ=2)

anova_results.index = [
    'age_group' if x == 'C(age_group)' else x
    for x in anova_results.index
]

print(anova_results.loc[['age_group']])


              sum_sq   df          F        PR(>F)
age_group  20.422743  2.0  17.597558  4.322550e-08


The one-way ANOVA test was performed to determine whether instructors' beauty scores differ across age groups. The calculated F-statistic was approximately 17.5976, with a p-value of 4.322549 × 10⁻⁸. Since the p-value is less than the significance level of 0.05, the null hypothesis is rejected. Therefore, there is a statistically significant difference in mean beauty scores among the three age groups. However, the ANOVA test alone does not identify which specific age groups differ significantly.

In [ ]:
model_q3 = smf.ols('eval ~ beauty', data=df).fit()

print(model_q3.summary())

print("\nBeauty coefficient:", model_q3.params['beauty'])
print("R-squared:", model_q3.rsquared)
print("t-statistic:", model_q3.tvalues['beauty'])
print("p-value:", model_q3.pvalues['beauty'])

                            OLS Regression Results                            
Dep. Variable:                   eval   R-squared:                       0.036
Model:                            OLS   Adj. R-squared:                  0.034
Method:                 Least Squares   F-statistic:                     17.08
Date:                Wed, 14 Oct 2026   Prob (F-statistic):           4.25e-05
Time:                        04:25:23   Log-Likelihood:                -375.32
No. Observations:                 463   AIC:                             754.6
Df Residuals:                     461   BIC:                             762.9
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.9983      0.025    157.727      0.0

The regression coefficient for beauty score is 0.1330, indicating a positive relationship between beauty score and teaching evaluation score. The p-value is approximately 0.00004247, which is less than the significance level of 0.05. Therefore, the null hypothesis is rejected, and there is a statistically significant positive association between beauty score and teaching evaluation score.

The R-squared value is 0.0357, indicating that beauty score explains approximately 3.57% of the variation in teaching evaluation scores.

Teaching evaluation scores are positively and significantly associated with beauty scores, although the relationship has relatively low explanatory power.